# Sequence Models for NLP

Language is inherently **sequential** — the meaning of a word depends on the words before and after it. *"The bank by the river"* and *"I deposited money at the bank"* use the same word in different senses, and only context resolves the ambiguity. Bag-of-words and TF-IDF discard word order entirely; **sequence models** process text token by token, building representations that capture temporal structure.

**Recurrent Neural Networks (RNNs)**, **LSTMs**, and **GRUs** were the dominant architecture for NLP before Transformers. They read sequences step by step, maintaining a hidden state that accumulates context. This notebook applies sequence models to NLP tasks — sentiment classification, sequence labeling, and machine translation — and shows how to represent, pad, and process text as sequential input.

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")
np.random.seed(42)

---

## 1. Why Sequences Matter for Language

Consider these sentences with identical word sets but different meanings:

- *"Dog bites man"* vs *"Man bites dog"*
- *"I didn't say he stole the money"* — placing emphasis on different words changes the implied meaning entirely.

Bag-of-words treats both versions identically. Sequence models preserve **word order** and **context**, enabling:

- Understanding negation (*"not good"* vs *"good"*).
- Resolving pronouns and coreference across a sentence.
- Modeling long-range dependencies (*"The keys that I left on the kitchen counter yesterday are missing"*).
- Generating fluent text one token at a time.

In [ ]:
def bag_of_words(text):
    return Counter(re.findall(r"\b[a-z]+\b", text.lower()))

s1 = "the man bites the dog"
s2 = "the dog bites the man"

bow1, bow2 = bag_of_words(s1), bag_of_words(s2)
print(f"Sentence 1: {s1}")
print(f"Sentence 2: {s2}")
print(f"\nBag-of-words identical: {bow1 == bow2}")
print(f"Word order different:   {s1.split() != s2.split()}")

---

## 2. Sequence Modeling Patterns in NLP

Different NLP tasks map to different input-output patterns:

| Pattern | Input → Output | NLP Task |
|---------|---------------|----------|
| **Many-to-one** | Full sentence → single label | Sentiment analysis, text classification |
| **One-to-many** | Single input → word sequence | Image captioning, text generation from prompt |
| **Many-to-many (aligned)** | Token sequence → tag sequence | POS tagging, NER |
| **Many-to-many (unaligned)** | Sequence → different length sequence | Machine translation, summarization |

The architecture choice — which RNN outputs to use, whether to run bidirectionally, whether to add an encoder-decoder — depends on the pattern.

---

## 3. Text as Sequences of Embeddings

A sequence model receives not raw words but **embedding vectors**. The pipeline:

1. **Tokenize** text into words or subwords.
2. **Map** each token to an integer index via a vocabulary.
3. **Look up** the embedding vector for each index.
4. **Pad** sequences to uniform length within a batch.
5. **Feed** the sequence of vectors into the RNN step by step.

```
"the cat sat" → [2, 5, 8] → [e₂, e₅, e₈] → RNN → output
```

The **embedding layer** is a learnable matrix $\mathbf{E} \in \mathbb{R}^{V \times d}$. During training, embeddings update alongside the RNN weights.

In [ ]:
def tokenize(text):
    return re.findall(r"\b[a-z]+\b", text.lower())

corpus = [
    "the cat sat on the mat",
    "the dog ran in the park",
    "a cat chased the dog",
]

vocab = {"<PAD>": 0, "<UNK>": 1}
for doc in corpus:
    for word in tokenize(doc):
        if word not in vocab:
            vocab[word] = len(vocab)

idx2word = {i: w for w, i in vocab.items()}
V, d = len(vocab), 8
E = np.random.randn(V, d) * 0.1  # embedding matrix

def encode(text):
    return [vocab.get(w, 1) for w in tokenize(text)]

def embed_sequence(indices):
    return np.array([E[i] for i in indices])

def pad_sequence(indices, max_len, pad_value=0):
    if len(indices) >= max_len:
        return indices[:max_len]
    return indices + [pad_value] * (max_len - len(indices))

text = "the cat sat on the mat"
indices = encode(text)
vectors = embed_sequence(indices)

print(f"Text:     {text}")
print(f"Indices:  {indices}")
print(f"Embedded: {vectors.shape}  (seq_len × embedding_dim)")
print(f"\nVocabulary ({V} tokens): {list(vocab.keys())[:10]}...")

---

## 4. The Vanilla RNN for Text

An RNN processes one token at a time, updating a hidden state:

$$\mathbf{h}_t = \tanh(\mathbf{W}_{hh} \mathbf{h}_{t-1} + \mathbf{W}_{xh} \mathbf{x}_t + \mathbf{b}_h)$$

For NLP, $\mathbf{x}_t$ is the embedding of the $t$-th word. The final hidden state $\mathbf{h}_T$ (or the full sequence of hidden states) feeds into a task-specific output layer.

For **many-to-one** classification, use the last hidden state: $\hat{y} = \text{softmax}(\mathbf{W}_{hy} \mathbf{h}_T)$.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def softmax(z):
    e = np.exp(z - z.max())
    return e / e.sum()

class VanillaRNN:
    """Minimal RNN for many-to-one text classification."""

    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        self.hidden_dim = hidden_dim
        self.E = np.random.randn(vocab_size, embed_dim) * 0.01
        self.W_xh = np.random.randn(hidden_dim, embed_dim) * 0.01
        self.W_hh = np.random.randn(hidden_dim, hidden_dim) * 0.01
        self.b_h = np.zeros(hidden_dim)
        self.W_hy = np.random.randn(num_classes, hidden_dim) * 0.01
        self.b_y = np.zeros(num_classes)

    def forward(self, indices):
        h = np.zeros(self.hidden_dim)
        hidden_states = []
        for idx in indices:
            if idx == 0:  # skip padding
                hidden_states.append(h.copy())
                continue
            x = self.E[idx]
            h = np.tanh(self.W_xh @ x + self.W_hh @ h + self.b_h)
            hidden_states.append(h.copy())
        logits = self.W_hy @ h + self.b_y
        return softmax(logits), hidden_states


rnn = VanillaRNN(vocab_size=V, embed_dim=8, hidden_dim=16, num_classes=2)
indices = encode("the cat sat on the mat")
probs, states = rnn.forward(indices)

print(f"Sequence length: {len(indices)}")
print(f"Hidden states:   {len(states)} vectors of dim {len(states[0])}")
print(f"Output probs:    class_0={probs[0]:.3f}, class_1={probs[1]:.3f}")

---

## 5. Many-to-One: Sentiment Classification

Sentiment analysis reads an entire sentence and outputs one label. The RNN processes each word, and the **final hidden state** summarizes the full sequence.

In [ ]:
sentiment_data = [
    ("this movie is fantastic and amazing", 1),
    ("wonderful acting great storyline", 1),
    ("best film ever seen", 1),
    ("loved every minute of it", 1),
    ("brilliant performance excellent work", 1),
    ("terrible movie waste of time", 0),
    ("awful acting boring plot", 0),
    ("worst film ever made", 0),
    ("horrible experience do not watch", 0),
    ("disappointed and frustrated completely", 0),
]

# Build vocabulary from sentiment corpus
sent_vocab = {"<PAD>": 0, "<UNK>": 1}
for text, _ in sentiment_data:
    for w in tokenize(text):
        if w not in sent_vocab:
            sent_vocab[w] = len(sent_vocab)

V_s = len(sent_vocab)
EMBED_DIM, HIDDEN_DIM = 16, 32


def train_sentiment_rnn(data, epochs=200, lr=0.05):
    model = VanillaRNN(V_s, EMBED_DIM, HIDDEN_DIM, num_classes=2)
    losses = []

    for epoch in range(epochs):
        total_loss = 0
        for text, label in data:
            indices = [sent_vocab.get(w, 1) for w in tokenize(text)]
            probs, _ = model.forward(indices)
            loss = -np.log(probs[label] + 1e-8)
            total_loss += loss

            # Simplified gradient update on output layer
            grad_logits = probs.copy()
            grad_logits[label] -= 1
            h_final = model.forward(indices)[1][-1]
            model.W_hy -= lr * np.outer(grad_logits, h_final)
            model.b_y -= lr * grad_logits

        losses.append(total_loss / len(data))

    return model, losses


model, losses = train_sentiment_rnn(sentiment_data, epochs=300, lr=0.1)

correct = 0
for text, label in sentiment_data:
    indices = [sent_vocab.get(w, 1) for w in tokenize(text)]
    pred = model.forward(indices)[0].argmax()
    correct += int(pred == label)

print(f"Training accuracy: {correct}/{len(sentiment_data)} = {correct/len(sentiment_data):.2f}")

plt.figure(figsize=(7, 3))
plt.plot(losses, color="steelblue")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("RNN sentiment training loss")
plt.tight_layout()
plt.show()

---

## 6. LSTM and GRU for Longer Sequences

Vanilla RNNs struggle with **vanishing gradients** over long sequences — early words in a sentence have little influence on the final hidden state. **LSTM** and **GRU** use gating mechanisms to preserve information across many time steps.

For NLP tasks involving long documents, complex syntax, or long-range dependencies, LSTM/GRU significantly outperform vanilla RNNs. They were the standard architecture for machine translation, language modeling, and speech recognition before Transformers.

| Model | Gates | Parameters | Best For |
|-------|-------|-----------|----------|
| Vanilla RNN | None | Fewest | Short sequences, teaching |
| GRU | Reset + update | Moderate | General purpose, faster training |
| LSTM | Forget + input + output | Most | Long sequences, complex dependencies |

---

## 7. Many-to-Many: Sequence Labeling

**Sequence labeling** assigns a tag to each token in the input. Applications include:

- **Part-of-speech (POS) tagging** — noun, verb, adjective, etc.
- **Named Entity Recognition (NER)** — person, organization, location.
- **Chunking** — identify noun phrases and verb phrases.

The RNN outputs a prediction at **every time step**: $\hat{y}_t = \text{softmax}(\mathbf{W}_{hy} \mathbf{h}_t)$. Input and output sequences have the same length.

In [ ]:
# POS tagging example
pos_data = [
    (["the", "cat", "sat"], ["DET", "NOUN", "VERB"]),
    (["a", "dog", "ran"], ["DET", "NOUN", "VERB"]),
    (["she", "reads", "books"], ["PRON", "VERB", "NOUN"]),
    (["he", "writes", "code"], ["PRON", "VERB", "NOUN"]),
    (["the", "quick", "fox"], ["DET", "ADJ", "NOUN"]),
    (["very", "happy", "today"], ["ADV", "ADJ", "NOUN"]),
]

pos_vocab = {"<PAD>": 0, "<UNK>": 1}
tag_vocab = {"<PAD>": 0}
for words, tags in pos_data:
    for w in words:
        if w not in pos_vocab:
            pos_vocab[w] = len(pos_vocab)
    for t in tags:
        if t not in tag_vocab:
            tag_vocab[t] = len(tag_vocab)

idx2tag = {i: t for t, i in tag_vocab.items()}


class SequenceLabeler:
    """RNN that outputs a tag at each time step."""

    def __init__(self, vocab_size, embed_dim, hidden_dim, num_tags):
        self.hidden_dim = hidden_dim
        self.E = np.random.randn(vocab_size, embed_dim) * 0.01
        self.W_xh = np.random.randn(hidden_dim, embed_dim) * 0.01
        self.W_hh = np.random.randn(hidden_dim, hidden_dim) * 0.01
        self.b_h = np.zeros(hidden_dim)
        self.W_hy = np.random.randn(num_tags, hidden_dim) * 0.01
        self.b_y = np.zeros(num_tags)

    def forward(self, word_indices):
        h = np.zeros(self.hidden_dim)
        tag_probs = []
        for idx in word_indices:
            x = self.E[idx]
            h = np.tanh(self.W_xh @ x + self.W_hh @ h + self.b_h)
            logits = self.W_hy @ h + self.b_y
            tag_probs.append(softmax(logits))
        return tag_probs

    def predict(self, word_indices):
        return [p.argmax() for p in self.forward(word_indices)]


labeler = SequenceLabeler(len(pos_vocab), 8, 16, len(tag_vocab))

# Train for a few epochs
for epoch in range(500):
    for words, tags in pos_data:
        w_idx = [pos_vocab.get(w, 1) for w in words]
        t_idx = [tag_vocab[t] for t in tags]
        probs = labeler.forward(w_idx)
        for t, (prob, true_tag) in enumerate(zip(probs, t_idx)):
            grad = prob.copy()
            grad[true_tag] -= 1
            labeler.W_hy -= 0.05 * np.outer(grad, np.ones(labeler.hidden_dim) * 0.1)
            labeler.b_y -= 0.05 * grad

print(f"{'Sentence':<25s} {'Predicted Tags'}")
print("-" * 55)
for words, true_tags in pos_data:
    w_idx = [pos_vocab.get(w, 1) for w in words]
    pred_tags = [idx2tag[i] for i in labeler.predict(w_idx)]
    sent = " ".join(words)
    print(f"{sent:<25s} {' / '.join(pred_tags)}")

---

## 8. Bidirectional RNNs for NLP

Forward-only RNNs see only **past** context. For sequence labeling, **future** words matter too — tagging *"bank"* as a location or organization requires seeing what follows.

A **bidirectional RNN** runs two RNNs:
- Forward: processes tokens left to right.
- Backward: processes tokens right to left.

At each position, hidden states are concatenated: $\mathbf{h}_t = [\overrightarrow{\mathbf{h}}_t; \overleftarrow{\mathbf{h}}_t]$.

Bidirectional models are standard for NER and POS tagging. They cannot be used for autoregressive generation (predicting the next word), where future tokens are unavailable.

In [ ]:
def bidirectional_rnn_forward(word_indices, embed_dim=8, hidden_dim=8):
    V = max(word_indices) + 1
    E = np.random.randn(V, embed_dim) * 0.01
    W_xh = np.random.randn(hidden_dim, embed_dim) * 0.01
    W_hh = np.random.randn(hidden_dim, hidden_dim) * 0.01

    # Forward pass
    h_fwd = np.zeros(hidden_dim)
    fwd_states = []
    for idx in word_indices:
        h_fwd = np.tanh(W_xh @ E[idx] + W_hh @ h_fwd)
        fwd_states.append(h_fwd.copy())

    # Backward pass
    h_bwd = np.zeros(hidden_dim)
    bwd_states = []
    for idx in reversed(word_indices):
        h_bwd = np.tanh(W_xh @ E[idx] + W_hh @ h_bwd)
        bwd_states.append(h_bwd.copy())
    bwd_states.reverse()

    # Concatenate at each position
    combined = [np.concatenate([f, b]) for f, b in zip(fwd_states, bwd_states)]
    return combined

words = ["Apple", "released", "a", "new", "iPhone"]
indices = [1, 2, 3, 4, 5]  # dummy indices
bi_states = bidirectional_rnn_forward(indices)

print(f"Sentence: {' '.join(words)}")
print(f"\nBidirectional hidden states (forward + backward concatenated):")
for w, h in zip(words, bi_states):
    print(f"  {w:12s} → dim={len(h)}  (fwd={len(h)//2} + bwd={len(h)//2})")

---

## 9. Sequence-to-Sequence for Translation

**Seq2seq** models handle input and output sequences of different lengths — the core architecture for machine translation, summarization, and dialogue.

```
Encoder:  "bonjour le monde" → context vector c
Decoder:  c → "hello" → "world" → <EOS>
```

1. The **encoder** RNN reads the source sentence and produces a context vector (final hidden state).
2. The **decoder** RNN generates the target sentence one token at a time, using the context vector and its previous outputs.

The context vector bottleneck limits performance on long sentences — addressed by the attention mechanism.

In [ ]:
# Character-level seq2seq for simple translation pairs
pairs = [
    ("hi", "hola"),
    ("cat", "gato"),
    ("dog", "perro"),
    ("yes", "si"),
    ("no", "no"),
]

chars = sorted(set(c for p in pairs for s in p for c in s + "<_>"))
char2idx = {c: i for i, c in enumerate(chars)}
idx2char = {i: c for c, i in char2idx.items()}

def encode_chars(word, add_eos=True):
    indices = [char2idx[c] for c in word]
    if add_eos:
        indices.append(char2idx[">"])
    return indices

print("Character-level encoding for seq2seq:")
for src, tgt in pairs[:3]:
    src_enc = encode_chars("<" + src)
    tgt_enc = encode_chars(tgt)
    src_chars = [idx2char[i] for i in src_enc]
    tgt_chars = [idx2char[i] for i in tgt_enc]
    print(f"  {src:5s} → {tgt:5s}")
    print(f"    encoder input:  {src_chars}")
    print(f"    decoder target: {tgt_chars}")

---

## 10. Attention Mechanism

The fixed-size context vector in early seq2seq models was a bottleneck — especially for long sentences. **Attention** (Bahdanau et al., 2015) lets the decoder look at **all encoder hidden states** at each decoding step, weighted by relevance.

$$\alpha_{tj} = \frac{\exp(\text{score}(\mathbf{h}_t^{\text{dec}}, \mathbf{h}_j^{\text{enc}}))}{\sum_k \exp(\text{score}(\mathbf{h}_t^{\text{dec}}, \mathbf{h}_k^{\text{enc}}))}$$

$$\mathbf{c}_t = \sum_j \alpha_{tj} \mathbf{h}_j^{\text{enc}}$$

The decoder receives a **context vector** $\mathbf{c}_t$ that is a weighted sum of encoder states — dynamically focused on the most relevant source words. When generating *"world"* from *"Bonjour le monde"*, attention weights peak on *"monde."*

Attention improved translation quality dramatically and led directly to **Transformers**, which use self-attention exclusively.

In [ ]:
def attention_weights(decoder_state, encoder_states):
    """Dot-product attention scores."""
    scores = np.array([decoder_state @ h for h in encoder_states])
    return softmax(scores)

# Simulated encoder states for "the cat sat" (3 words)
np.random.seed(0)
encoder_states = [np.random.randn(8) for _ in range(3)]
decoder_state = np.random.randn(8)

weights = attention_weights(decoder_state, encoder_states)
source_words = ["the", "cat", "sat"]

print("Attention weights (decoder looking at encoder):")
for w, a in zip(source_words, weights):
    bar = "█" * int(a * 40)
    print(f"  {w:5s} {a:.3f}  {bar}")

context = sum(a * h for a, h in zip(weights, encoder_states))
print(f"\nContext vector shape: {context.shape}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(source_words, weights, color="steelblue")
ax.set_ylabel("Attention weight")
ax.set_title("Decoder attention over encoder tokens")
plt.tight_layout()
plt.show()

---

## 11. Text Generation with RNNs

**One-to-many** generation: given a starting token (or empty state), the RNN predicts the next token, feeds it back as input, and repeats until an end-of-sequence token is produced.

$$P(w_t \mid w_1, \ldots, w_{t-1}) = \text{softmax}(\mathbf{W}_{hy} \mathbf{h}_t)$$

This **autoregressive** process generates text one character or word at a time. Training uses **teacher forcing** — feeding the correct previous token during training rather than the model's own (possibly wrong) prediction.

In [ ]:
def generate_text(model, char2idx, idx2char, seed="<", max_len=20, temperature=1.0):
    """Autoregressive character generation from a trained RNN."""
    indices = [char2idx.get(c, 0) for c in seed]
    generated = list(seed)

    for _ in range(max_len):
        probs, _ = model.forward(indices)
        # Use output as next-char distribution (simplified)
        probs = probs / temperature
        probs = probs / probs.sum()
        next_idx = np.random.choice(len(probs), p=probs)
        if idx2char.get(next_idx, ">") == ">":
            break
        generated.append(idx2char.get(next_idx, "?"))
        indices.append(next_idx)

    return "".join(generated)

print("Autoregressive generation process:")
print("  1. Start with seed token '<'")
print("  2. RNN forward → probability over vocabulary")
print("  3. Sample next token (temperature controls randomness)")
print("  4. Append token, feed back as input")
print("  5. Repeat until '<EOS>' or max length")
print()
print("Temperature effects:")
print("  T=0.1 → nearly deterministic (always pick highest prob)")
print("  T=1.0 → sample from model distribution")
print("  T=2.0 → more random, diverse but less coherent")

---

## 12. Word-Level vs Character-Level

| Level | Vocabulary | Sequence Length | Best For |
|-------|-----------|----------------|----------|
| **Word** | 10K–100K words | Shorter | Classification, translation, QA |
| **Character** | ~100 characters | Much longer | Morphologically rich languages, typos, small data |
| **Subword** | 30K–100K pieces | Medium | Modern LLMs; handles OOV |

Word-level models are faster but cannot handle unknown words. Character-level models have tiny vocabularies and no OOV problem but must learn word structure from scratch. Subword tokenization (BPE, WordPiece) combines both advantages — the standard for modern systems.

---

## 13. Handling Variable-Length Sequences

Sentences have different lengths. Batching requires consistent dimensions:

**Padding** — add `<PAD>` tokens to shorter sequences.

**Packing** — in PyTorch, `pack_padded_sequence` skips pad tokens during RNN computation for efficiency.

**Masking** — attention models use masks so padded positions receive zero weight.

**Truncation** — cut sequences exceeding a maximum length (common with BERT's 512-token limit).

Always track sequence lengths and mask pad tokens in loss computation — otherwise the model learns from meaningless padding.

In [ ]:
batch = [
    encode("the cat sat"),
    encode("the dog ran in the park"),
    encode("a cat"),
]
lengths = [len(s) for s in batch]
max_len = max(lengths)

padded = np.array([pad_sequence(s, max_len) for s in batch])
mask = np.array([[1]*l + [0]*(max_len-l) for l in lengths])

print(f"Sequence lengths: {lengths}")
print(f"Padded batch shape: {padded.shape}")
print(f"\nPadded indices:\n{padded}")
print(f"\nAttention mask (1=real, 0=pad):\n{mask}")

---

## 14. Sequence Models vs Bag-of-Words

| Aspect | BoW / TF-IDF | Sequence Models (RNN/LSTM) |
|--------|-------------|---------------------------|
| **Word order** | Ignored | Preserved |
| **Negation** | "not good" ≈ "good" | Captured |
| **Long-range deps** | Not modeled | LSTM handles moderate length |
| **Training speed** | Very fast | Slower (sequential steps) |
| **Data needed** | Small datasets OK | More data typically needed |
| **Interpretability** | Feature weights | Hidden states opaque |

For many classification tasks with limited data, TF-IDF + logistic regression remains competitive. Sequence models shine when word order, context, and generation matter.

---

## 15. From RNNs to Transformers

RNNs dominated NLP from 2014–2018. Limitations drove the shift to Transformers:

- **Sequential processing** — cannot parallelize across time steps.
- **Vanishing gradients** — even LSTMs struggle with very long sequences.
- **Context bottleneck** — compressing a sentence into one vector loses information.

Transformers replaced recurrence with **self-attention** — every token attends to every other token in parallel. They train faster, scale to longer contexts, and achieve superior performance on virtually all NLP benchmarks.

RNN concepts remain relevant: embedding layers, sequence padding, encoder-decoder structure, and attention all carry forward into Transformer architectures.

---

## 16. Summary

Language is sequential — word order and context determine meaning. **Sequence models** (RNNs, LSTMs, GRUs) process text token by token, maintaining hidden state that accumulates context.

NLP tasks map to different patterns: **many-to-one** for classification, **many-to-many** for tagging, **seq2seq** for translation. **Bidirectional RNNs** capture both past and future context for labeling tasks. **Attention** lets decoders focus on relevant encoder states, bridging seq2seq to Transformers.

Text enters as **embedding sequences**, padded for batching. While Transformers have superseded RNNs for most tasks, sequence model concepts — embeddings, encoder-decoders, attention — remain foundational to modern NLP.